# Stage 27A: XY spatial-surface state audit

既知prefixのU=TVT+ZをXY平面として外挿し、強いbaseへの小さな補正が再現可能か監査します。学習・提出・予約120 wellsの使用はありません。CPUで実行してください。


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json,os,shutil,subprocess,sys
REPOSITORY_URL='https://github.com/Okada-N13/rogii.git'
repo_dir=Path('/content/ROGII'); drive_root=Path('/content/drive/MyDrive/kaggle/rogii')
artifact_dir=drive_root/'artifacts'; data_dir=drive_root/'data'
if not (repo_dir/'.git').is_dir(): subprocess.run(['git','clone',REPOSITORY_URL,str(repo_dir)],check=True)
else: subprocess.run(['git','-C',str(repo_dir),'pull','--ff-only','origin','main'],check=True)
if shutil.which('uv') is None: subprocess.run(['bash','-lc','curl -LsSf https://astral.sh/uv/install.sh | sh'],check=True)
os.environ['PATH']='/root/.local/bin:'+os.environ['PATH']
subprocess.run(['uv','sync','--frozen'],cwd=repo_dir,check=True)
assert (data_dir/'train').is_dir(),data_dir
print('Stage 27A uses CPU and preserves all 120 confirmation wells')


In [ ]:
stage16b_run=artifact_dir/'stage16b_testlike_validation_full_v003'
stage17a_run=artifact_dir/'stage17_public_replay_full_v002'
public_oof_run=artifact_dir/'stage7_public_residual_gate_full_v001'
stage21b_run=artifact_dir/'stage21b_prefix_confidence_full_v001'
stage24a_run=artifact_dir/'stage24a_scaled_ordinal_emission_full_v001'
required=[stage16b_run/'well_assignments.parquet',stage17a_run/'cut_report.parquet',public_oof_run/'base_oof.parquet',stage21b_run/'confidence_cut_report.parquet',stage24a_run/'summary.json']
for path in required: assert path.is_file(),path
print(*required,sep='\n')


In [ ]:
RUN_ID='stage27a_spatial_surface_state_full_v001'; run_dir=artifact_dir/RUN_ID
if run_dir.exists() and not (run_dir/'summary.json').is_file():
    resolved=run_dir.resolve(); expected=(artifact_dir/RUN_ID).resolve()
    assert resolved==expected and resolved.parent==artifact_dir.resolve(),resolved
    print('Removing incomplete prior run:',resolved); shutil.rmtree(resolved)
if not (run_dir/'summary.json').is_file():
    command=[sys.executable,'-m','rogii.cli.spatial_surface_state','--config','configs/experiment/stage27a_spatial_surface_state.yaml','--stage16b-run',str(stage16b_run),'--stage17a-run',str(stage17a_run),'--public-oof-run',str(public_oof_run),'--stage24a-run',str(stage24a_run),'--validation-run',str(stage21b_run),'--data-dir',str(data_dir),'--artifact-dir',str(artifact_dir),'--run-id',RUN_ID]
    audit_env=os.environ.copy(); audit_env['PYTHONPATH']=str(repo_dir/'src')+':'+audit_env.get('PYTHONPATH','')
    result=subprocess.run(command,cwd=repo_dir,env=audit_env,text=True,capture_output=True)
    if result.stdout: print(result.stdout)
    if result.stderr: print(result.stderr)
    if result.returncode: raise RuntimeError(f'Stage 27A failed: {command}')
summary=json.loads((run_dir/'summary.json').read_text())
{key:summary[key] for key in ['stage27a_complete','promoted_to_stage27b_spatial_surface_model','cuts','wells','rows','base_rmse','oracle_rmse','oracle_delta','primary_profile','primary_metrics','bootstrap_95pct','end_u_change_correlations','gates','reserved_confirmation_used','next_step']}


In [ ]:
import pandas as pd
pd.DataFrame(summary['profile_report']).sort_values('rmse_delta')


最後のsummary辞書とprofile表を共有してください。primary以外は診断専用で、同じvalidation上の最良値へ事後変更しません。
